In [2]:
import logging
import warnings
import torch
import torchvision.transforms as transforms
import importlib
from components import FL_sim
from components.other_utilities import models_to_train
import components.broadcast_components.broadcasting_process.broadcast_reporting_utilities
import components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol
import components.broadcast_components.compressor.rans_coding
import components.other_utilities.datasets

importlib.reload(components.broadcast_components.compressor.rans_coding)
importlib.reload(components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol)
importlib.reload(components.broadcast_components.WZ_models.wz_quant_ANN)
importlib.reload(components.broadcast_components.WZ_models.wz_quant_RNN)
importlib.reload(components.broadcast_components.broadcasting_process.broadcast_reporting_utilities)
importlib.reload(FL_sim)
importlib.reload(models_to_train)

from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator
from components.other_utilities.datasets import FasterSVHN, FasterImageNet
from components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol import WZServerTrainingPerRoundProtocol
from components.broadcast_components.broadcasting_process.broadcast_reporting_utilities import BroadcastMetricGatheringUtilities
from components.broadcast_components.WZ_models.wz_quant_RNN import PL_EncoderDecoder_RNN

torch.set_float32_matmul_precision('high')
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", "LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]")
warnings.filterwarnings("ignore", "You defined a `validation_step` but")
warnings.filterwarnings("ignore", "Starting from v1.9.0, `tensorboardX` has been")
warnings.filterwarnings("ignore", "The number of training batches")
warnings.filterwarnings("ignore", "`Trainer.fit` stopped: ")

ModuleNotFoundError: No module named 'lightning'

In [1]:

dataset = [
    FasterImageNet(
        root='./data/ImageNet', split=s,
        transform=transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])
    ) for s in ['train', 'val']
]


# dataset = [
#     FasterSVHN(
#         root='./data/SVHN', split=s,
#         transform=transforms.Compose([
#             transforms.Resize(32),
#             transforms.ToTensor(),
#             transforms.Normalize(
#                 mean=[0.4377, 0.4438, 0.4728],
#                 std=[0.1980, 0.2010, 0.1970]
#             ),
#         ])
#     ) for s in ['train', 'test']]

# dataset = [torch.utils.data.Subset(d, list(range(100))) for d in dataset]
# for i in range(10):
#     for d in dataset:
#         d.dataset.labels[i]=i

NameError: name 'FasterImageNet' is not defined

In [3]:
worker_count = 5
batch_size = 7500

# *****************

user_logger = None#UnifiedLoggingClass(worker_count)
# ****
wz_model = PL_EncoderDecoder_RNN(inp_dim=1, side_info_size=0, num_planes=3, bins_per_plane=4, tau=5, reconst_ld=3.5, lr=8e-4, ).to(torch.float32)
# ****
path_to_basic = r'/data/basicRNN_2plane_4bins_state.pt'
wz_model.load_state_dict(torch.load(path_to_basic, map_location='cpu'))
# ****
base_quantizer = WZQuantizer(wz_model, train_sample_size=100_000,
        count_side_info_data=0, enable_progress_bar=False, user_logger=user_logger)
broadcast_prot_base = WZServerTrainingPerRoundProtocol(worker_count, base_quantizer)
broadcast_prot = BroadcastMetricGatheringUtilities(broadcast_prot_base, user_logger=user_logger)

# *****************

model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.005,)
model.load_state_dict(torch.load('data/resnet18_svhn.pth', map_location='cpu'))

# *****************
sim = FLSimulator(
    pl_model=model, num_agents=worker_count, communication_rounds=50, client_epochs_per_round=10,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False, user_logger=user_logger)
# ****
sim.run_simulation(broadcast_prot)


round 1/50 --------------------
  - reporting global model metrics
         - train> loss: 2.18, auc: 0.61    |    test> loss: 2.17, auc: 0.61



LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


      > training agent 1/5
             + initiating broadcast


D:\User\App Files\Projects\VUB-ACS-25_Thesis\components\broadcast_components\broadcasting_process\WZ_broadcast.py:58: RuntimeWarning: overflow encountered in cast
  return obj.astype(numpy_dtype)


TypingError: [1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1m[1m- Resolution failure for literal arguments:
[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1mNo implementation of function Function(<built-in function getitem>) found for signature:

 >>> getitem(list(int64)<iv=None>, array(uint8, 1d, C))

There are 22 candidate implementations:
[1m   - Of which 22 did not match due to:
   Overload of function 'getitem': File: <numerous>: Line N/A.
     With argument(s): '(list(int64)<iv=None>, array(uint8, 1d, C))':[0m
[1m    No match.[0m
[0m
[0m[1mDuring: typing of intrinsic-call at D:\User\Software\Miniconda3\envs\UniversalCondaEnv\Lib\site-packages\rans\rANSCoder.py (81)[0m
[1m
File "..\..\..\Software\Miniconda3\envs\UniversalCondaEnv\Lib\site-packages\rans\rANSCoder.py", line 81:[0m
[1m    def encode_symbol(self, freqs, symbol):
        <source elided>
        (pdf, cdf) = float_to_int_probs(freqs)
[1m        freq =